In [ ]:
# ruff: noqa: T201, T203, ERA001, BLE001, E402
from __future__ import annotations

import logging
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

while Path.cwd().name in ["notebooks", "preprocessing"]:
    os.chdir("..")

from src.constants import PAIRS_SEPARATOR
from src.evaluate import (
    get_gt_pairs,
)
from src.formatting import (
    format_gt_pairs_filepath,
    format_oracle_pairs_filepath,
    format_oracle_pairs_precomputed_dir,
    format_subsets_ontologies_paths,
)
from src.onto_access import OntologyAccess
from src.processing import save_oracle_pairs_with_prompts
from src.prompts.prompts import (
    prompt_direct_entity,
    prompt_direct_entity_ontological,
    prompt_direct_entity_with_synonyms,
    prompt_sequential_hierarchy,
    prompt_sequential_hierarchy_ontological,
    prompt_sequential_hierarchy_with_synonyms,
)

pd.set_option("display.max_rows", None)
logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

In [ ]:
PROMPT_FUNCTIONS_MAP = {
    prompt_function.__name__.replace("prompt_", ""): prompt_function
    for prompt_function in [
        prompt_direct_entity,
        prompt_direct_entity_ontological,
        prompt_direct_entity_with_synonyms,
        prompt_sequential_hierarchy,
        prompt_sequential_hierarchy_ontological,
        prompt_sequential_hierarchy_with_synonyms,
    ]
}

In [ ]:
ALL_DATASET_NAMES = {
    "anatomy": ["human-mouse"],
    "bioml-2024": ["snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm", "ncit-doid", "omim-ordo"][-2:-1],
    # "largebio": ["fma-snomed", "snomed-nci", "fma-nci"],
    # "largebio_small": ["fma-nci", "snomed-nci", "fma-nci"],
}
random_state = 42

for dataset_name, set_names in ALL_DATASET_NAMES.items():
    for set_name in set_names:
        # read original pairs
        pairs_path = format_oracle_pairs_filepath(dataset_name, set_name)
        base_pairs_df = pd.read_csv(pairs_path, sep=PAIRS_SEPARATOR, header=None)
        original_pairs_set = set(zip(base_pairs_df[0], base_pairs_df[1]))

        # prepared shuffled pairs
        a_shuffled = base_pairs_df[0].sample(frac=1, random_state=random_state).reset_index(drop=True)
        b_shuffled = base_pairs_df[1].sample(frac=1, random_state=random_state + 1).reset_index(drop=True)
        false_candidates_pairs = set(zip(a_shuffled, b_shuffled))

        # filter out original pairs
        print(
            "Before filtering original pairs:",
            f"num_candidates={len(false_candidates_pairs)}",
            f"num_original={len(original_pairs_set)}",
            f"overlap={len(original_pairs_set & false_candidates_pairs)}",
        )
        false_candidates_pairs -= original_pairs_set

        # read groudn truth file pairs
        gt_positive_pairs_path = format_gt_pairs_filepath(dataset_name, set_name)
        gt_df = get_gt_pairs(gt_positive_pairs_path)
        gt_pairs = {(row["Source"], row["Target"]) for _, row in gt_df.iterrows()}

        # filter out gt pairs
        print(
            "Before filtering gt pairs:",
            f"num_candidates={len(false_candidates_pairs)}",
            f"num_gt={len(gt_df)}",
            f"do_overlap={any(x in gt_pairs for x in false_candidates_pairs)}",
        )
        false_candidates_pairs = false_candidates_pairs - gt_pairs
        print(
            "After filtering:",
            f"num_false_candidates={len(false_candidates_pairs)}",
            f"do_overlap={any(x in gt_pairs for x in false_candidates_pairs)}",
        )

        # prepare final df
        new_pairs_rows = [(pair[0], pair[1], "-", "-", "-") for pair in false_candidates_pairs]
        new_pairs_df = pd.DataFrame(new_pairs_rows)

        print("Number of synthetic negative pairs generated:", len(new_pairs_rows))

        negative_pairs_path = pairs_path.parent / "synthetic_negative_pairs.csv"
        new_pairs_df.to_csv(negative_pairs_path, index=False, header=None, sep=PAIRS_SEPARATOR)


In [ ]:
ALL_DATASET_NAMES = {
    "anatomy": ["human-mouse"],
    "bioml-2024": ["snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm", "ncit-doid", "omim-ordo"][-2:-1],
    # "largebio": ["fma-snomed", "snomed-nci", "fma-nci"],
    # "largebio_small": ["fma-nci", "snomed-nci", "fma-nci"],
}
prompt_functions = list(PROMPT_FUNCTIONS_MAP.values())

for dataset_name, set_names in ALL_DATASET_NAMES.items():
    for set_name in set_names:
        try:
            ontologies = []
            for onto_path in tqdm(
                format_subsets_ontologies_paths(dataset_name, set_name),
                desc=f"Loading ontologies: {dataset_name} - {set_name}",
            ):
                ontologies.append(OntologyAccess(onto_path, annotate_on_init=True))

            pairs_path = format_oracle_pairs_filepath(dataset_name, set_name)
            negative_pairs_path = pairs_path.parent / "synthetic_negative_pairs.csv"
            oracle_negative_pairs_dir = format_oracle_pairs_precomputed_dir(
                dataset_name, set_name, suffix="_synthetic_negative"
            )

            pairs_with_prompts_df = save_oracle_pairs_with_prompts(
                negative_pairs_path,
                *ontologies,
                prompt_functions,
                oracle_negative_pairs_dir,
                PAIRS_SEPARATOR,
                max_workers=1,
            )
        except Exception as e:
            print(f"Error processing {dataset_name} - {set_name}: {e}")
            continue